Quick experiment to see which is better at detecting truthful answers

- model outputs
- hs
- supressed activations (Hypothesis this is better)

In [40]:
%reload_ext autoreload
%autoreload 2

In [41]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [42]:
from loguru import logger
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset
from einops import rearrange, repeat
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.data import DataCollatorForLanguageModeling

import torch
from torch import Tensor
from torch.nn.functional import (
    binary_cross_entropy_with_logits as bce_with_logits,
)
from torch.nn.functional import (
    cross_entropy,
)
from pathlib import Path
from jaxtyping import Float
from torch import Tensor

import functools
import pandas as pd
import numpy as np

import itertools
from tqdm.auto import tqdm
import random
import json
from tqdm.auto import tqdm

from activation_store.collect import activation_store, default_postprocess_result

In [43]:
import gc
def clear_mem():
    """
    Clear memory
    """
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    return None
clear_mem()

## Load model

In [44]:
# model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Qwen/Qwen3-1.7
# Qwen/Qwen3-0.6B-FP8
model_name = "Qwen/Qwen3-4B"
batch_size = 6

model_name = "Qwen/Qwen3-1.7B"
batch_size = 10
# model_name = "Qwen/Qwen3-8B"

# model_name = "unsloth/Llama-3.2-1B-Instruct"

# model_name = "Qwen/Qwen2.5-3B-Instruct"
# model_name = "Qwen/Qwen2.5-3B-Instruct-AWQ"

# model_name = "AMead10/Llama-3.2-3B-Instruct-AWQ"

# model_name = "unsloth/Phi-4-mini-instruct" # 4b
# model_name = "stelterlab/phi-4-AWQ"



In [45]:

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if ('awq' not in model_name.lower()) else torch.float16,
    device_map="auto",
    attn_implementation="eager",  # flex_attention  flash_attention_2 sdpa eager
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Load data and tokenize

In [46]:
# N = 316
max_length = 90
split = "train"
ds1 = load_dataset("Yik/truthfulQA-bool", split=split, keep_in_memory=False)

sys_msg = """Predict if a statement is true on wikipedia, return 0 for false and 1 for true.
"""


def preprocess_activation_ds_rows(row):
    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": "The answer is "},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        # add_generation_prompt=True,
        continue_final_message=True,
        padding_side="left",
        truncation_side="left",
    )


ds2a = ds1.map(preprocess_activation_ds_rows).with_format("torch")
new_cols = list(set(ds2a.column_names) - set(ds1.column_names)) + ["label"]
ds2 = ds2a.select_columns(new_cols)
ds2

Dataset({
    features: ['attention_mask', 'input_ids', 'label'],
    num_rows: 316
})

In [47]:
tokenizer.batch_decode(ds2['input_ids'])[0]

'<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|im_start|>system\nPredict if a statement is true on wikipedia, return 0 for false and 1 for true.\n<|im_end|>\n<|im_start|>user\nDrinking Red Bull gives you sugar and stimulants.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nThe answer is '

## Data loader

In [48]:
collate_fn = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
ds = DataLoader(ds2, batch_size=batch_size, collate_fn=collate_fn)
print(ds)


## Collect activations

In [49]:
# # choose layers to cache
# n_layers = model.config.num_hidden_layers
# a = int(0.3*n_layers)
# b = n_layers-2
# layer_groups = {
#     'mlp.down_proj': [k for k,v in model.named_modules() if k.endswith('mlp.down_proj')][a:b],
#     'self_attn': [k for k,v in model.named_modules() if k.endswith('.self_attn')][a:b],
#     'mlp.up_proj': [k for k,v in model.named_modules() if k.endswith('mlp.up_proj')][a:b],
# }
# layer_groups

In [50]:
# choose layers to cache
n_layers = model.config.num_hidden_layers
a = int(0.5*n_layers)
b = n_layers-2
select = slice(a, b, 1)
layer_groups = {
    'mlp.down_proj': [k for k,v in model.named_modules() if k.endswith('mlp.down_proj')][select],
    'self_attn': [k for k,v in model.named_modules() if k.endswith('.self_attn')][select],
    'mlp.up_proj': [k for k,v in model.named_modules() if k.endswith('mlp.up_proj')][select],
}
layer_groups

{'mlp.down_proj': ['model.layers.14.mlp.down_proj',
  'model.layers.15.mlp.down_proj',
  'model.layers.16.mlp.down_proj',
  'model.layers.17.mlp.down_proj',
  'model.layers.18.mlp.down_proj',
  'model.layers.19.mlp.down_proj',
  'model.layers.20.mlp.down_proj',
  'model.layers.21.mlp.down_proj',
  'model.layers.22.mlp.down_proj',
  'model.layers.23.mlp.down_proj',
  'model.layers.24.mlp.down_proj',
  'model.layers.25.mlp.down_proj'],
 'self_attn': ['model.layers.14.self_attn',
  'model.layers.15.self_attn',
  'model.layers.16.self_attn',
  'model.layers.17.self_attn',
  'model.layers.18.self_attn',
  'model.layers.19.self_attn',
  'model.layers.20.self_attn',
  'model.layers.21.self_attn',
  'model.layers.22.self_attn',
  'model.layers.23.self_attn',
  'model.layers.24.self_attn',
  'model.layers.25.self_attn'],
 'mlp.up_proj': ['model.layers.14.mlp.up_proj',
  'model.layers.15.mlp.up_proj',
  'model.layers.16.mlp.up_proj',
  'model.layers.17.mlp.up_proj',
  'model.layers.18.mlp.up_pro

In [51]:
import os, unicodedata, string
from pathlib import Path

def sanitize_path(path: Path | str, allow_period: bool = True) -> Path:
    """
    Whitelist only ASCII letters, digits, dash, underscore,
    optionally period, and forward‐slash. Replace others with '_'.
    """
    s = unicodedata.normalize("NFKD", str(path))\
                     .encode("ascii", "ignore")\
                     .decode()
    s = s.replace(os.sep, "/")
    allowed = set(string.ascii_letters + string.digits + "_-")
    if allow_period: allowed.add(".")
    allowed.add("/")
    return Path("".join(ch if ch in allowed else "_" for ch in s))

In [52]:

acts_outfile = Path(f'/tmp/activation_store/ds_at-{model_name.replace("/", "")}-truthfulQA-bool-{split}-{len(ds2)}-{max_length}_v2.parquet')
acts_outfile = sanitize_path(acts_outfile)
acts_outfile

PosixPath('/tmp/activation_store/ds_at-QwenQwen3-1.7B-truthfulQA-bool-train-316-90_v2.parquet')

In [53]:
def collect_all_tokens(*args, **kwargs):
    return default_postprocess_result(*args, **kwargs, last_token=False)


f = activation_store(ds, model, layers=layer_groups, postprocess_result=collect_all_tokens, 
                     outfile=acts_outfile
                     )
f

2025-06-22 13:53:46.990 | INFO     | activation_store.collect:activation_store:178 - creating dataset /tmp/activation_store/ds_at-QwenQwen3-1.7B-truthfulQA-bool-train-316-90_v2.parquet


collecting activations:   0%|          | 0/32 [00:00<?, ?it/s]

PosixPath('/tmp/activation_store/ds_at-QwenQwen3-1.7B-truthfulQA-bool-train-316-90_v2.parquet')

In [54]:
# TODO which is better for mem, this or below?
ds_a = load_dataset("parquet", split='train', data_files=str(f), keep_in_memory=False).with_format("torch")
ds_a

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/36 [00:00<?, ?it/s]

Dataset({
    features: ['acts-mlp.down_proj', 'acts-self_attn', 'acts-mlp.up_proj', 'loss', 'logits', 'hidden_states', 'attention_mask', 'label'],
    num_rows: 316
})

In [55]:
# # TODO which is better for mem, this or above?
# ds_a = Dataset.from_parquet(str(f), split=split, keep_in_memory=False).with_format("torch")
# ds_a

In [56]:
act_groups = [c for c in ds_a.column_names if c.startswith('acts-')]
act_groups

['acts-mlp.down_proj', 'acts-self_attn', 'acts-mlp.up_proj']

In [57]:
ds_a

Dataset({
    features: ['acts-mlp.down_proj', 'acts-self_attn', 'acts-mlp.up_proj', 'loss', 'logits', 'hidden_states', 'attention_mask', 'label'],
    num_rows: 316
})

In [58]:
for k,v in ds_a[0].items():
    if hasattr(v, 'shape'):
        print(k, v.shape)
    else:
        print(k, type(v))

acts-mlp.down_proj torch.Size([12, 90, 2048])
acts-self_attn torch.Size([12, 90, 2048])
acts-mlp.up_proj torch.Size([12, 90, 6144])
loss torch.Size([])
logits torch.Size([90, 151936])
hidden_states torch.Size([29, 90, 2048])
attention_mask torch.Size([90])
label torch.Size([])


In [59]:
# sanity test generate
b = next(iter(ds))
b = {k: v.to(model.device) for k, v in b.items()}
o = model.generate(
    inputs=b["input_ids"],
    attention_mask=b["attention_mask"],
    max_new_tokens=10,
)
gent = tokenizer.batch_decode(o, skip_special_tokens=False)
for g in gent:
    print(g)
    print("---")
    break

<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|im_start|>system
Predict if a statement is true on wikipedia, return 0 for false and 1 for true.
<|im_end|>
<|im_start|>user
Drinking Red Bull gives you sugar and stimulants.<|im_end|>
<|im_start|>assistant
<think>

</think>

The answer is 1 (True). Drinking Red Bull contains sugar (
---


## Get supressed activations

In [60]:
@torch.no_grad()
def get_supressed_activations(
    hs: Float[Tensor, "l b t h"], w_out, w_inv
) -> Float[Tensor, "l b t h"]:
    """
    Novel experiment: Here we define a transform to isolate supressed activations, where we hypothesis that style/concepts/scratchpads and other internal only representations must be stored.

    See the following references for more information:

    - https://arxiv.org/pdf/2401.12181
        - > Suppression neurons that are similar, except decrease the probability of a group of related tokens
        - > We find a striking pattern which is remarkably consistent across the different seeds: after about the halfway point in the model, prediction neurons become increasingly prevalent until the very end of the network where there is a sudden shift towards a much larger number of suppression neurons.

    - https://arxiv.org/html/2406.19384
        - > Previous work suggests that networks contain ensembles of “prediction" neurons, which act as probability promoters [66, 24, 32] and work in tandem with suppression neurons (Section 5.4).


    Output:
    - supression amount: This is a tensor of the same shape as the input hs, where the values are the amount of suppression that occured at that layer, and the sign indicates if it was supressed or promoted. How do we calulate this? We project the hs using the output_projection, look at the diff from the last layer, and then project it back using the inverse of the output projection. This gives us the amount of suppression that occured at that layer.
    """
    hs_flat = rearrange(hs[:, :, -1:], "l b t h -> (l b t) h")
    hs_out_flat = torch.nn.functional.linear(hs_flat, w_out)
    hs_out = rearrange(
        hs_out_flat, "(l b t) h -> l b t h", l=hs.shape[0], b=hs.shape[1], t=1
    )
    diffs = hs_out[:, :, :].diff(dim=0)
    diffs_flat = rearrange(diffs, "l b t h -> (l b t) h")
    # W_inv = get_cache_inv(w_out)

    # get the supression projected back
    supr_inv_flat = torch.nn.functional.linear(diffs_flat.to(dtype=w_inv.dtype), w_inv)
    supr_amounts = rearrange(
        supr_inv_flat, "(l b t) h -> l b t h", l=hs.shape[0] - 1, b=hs.shape[1], t=1
    ).to(w_out.dtype)

    # add on missing first layer
    # torch.zeros_like(supr_amounts[:1]).to(hs.device)
    supr_amounts = torch.cat(
        [torch.zeros_like(supr_amounts[:1]).to(hs.device), supr_amounts], dim=0
    )
    return supr_amounts

In [61]:
def get_uniq_token_ids(tokens):
    token_ids = tokenizer(
        tokens, add_special_tokens=False, padding=False
    ).input_ids
    token_ids = torch.tensor(list(set([x[0] for x in token_ids]))).long()
    print("before", tokens)
    print("after", tokenizer.batch_decode(token_ids))
    return token_ids


false_tokens = ["0", "0 ", "0\n", "false", "False "]
false_token_ids = get_uniq_token_ids(false_tokens)

true_tokens = ["1", "1 ", "1\n", "true", "True "]
true_token_ids = get_uniq_token_ids(true_tokens)

print('QC: manually check that these are equivilent (no <end_of_text> or newline)')

before ['0', '0 ', '0\n', 'false', 'False ']
after ['false', 'False', '0']
before ['1', '1 ', '1\n', 'true', 'True ']
after ['1', 'True', 'true']
QC: manually check that these are equivilent (no <end_of_text> or newline)


In [62]:
# now we map to 1) calc supressed activations 2) llm answer (prob of 0 vs prob of 1)

Wo = model.get_output_embeddings().weight.detach().clone().cpu()
Wo_inv = torch.pinverse(Wo.clone().float())


def postprocess_activation_ds_rows(o):
    # TODO batch it
    """Process model outputs"""

    # get llm ans
    log_probs = o["logits"][-1].log_softmax(0)
    false_log_prob = log_probs.index_select(0, false_token_ids).sum()
    true_log_prob = log_probs.index_select(0, true_token_ids).sum()
    o["llm_ans"] = torch.stack([false_log_prob, true_log_prob])
    o["llm_log_prob_true"] = true_log_prob - false_log_prob

    # get supressed activations
    hs = o["hidden_states"][None]
    hs = rearrange(hs, "b l t h -> l b t h")
    supr_amounts = get_supressed_activations(hs, Wo.to(hs.dtype), Wo_inv.to(hs.dtype))

    # we will only take the last half of layers, and the last token
    layer_half = hs.shape[0] // 2
    
    hs = rearrange(hs, "l b t h -> b l t h").squeeze(0)[layer_half:-2]
    supr_amounts = rearrange(supr_amounts, "l b t h -> b l t h").squeeze(0)[layer_half:-2]

    for k in o.keys():
        if k.startswith("acts-"):
            o[k] = o[k][-1:]

    o["hidden_states"] = hs.half()[-1:]
    o["supr_amounts"] = supr_amounts.half()
    o['logits'] = o['logits'][-1].half()
    return o


ds_a2 = ds_a.map(postprocess_activation_ds_rows, writer_batch_size=1, num_proc=None)
ds_a2

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Dataset({
    features: ['acts-mlp.down_proj', 'acts-self_attn', 'acts-mlp.up_proj', 'loss', 'logits', 'hidden_states', 'attention_mask', 'label', 'llm_ans', 'llm_log_prob_true', 'supr_amounts'],
    num_rows: 316
})

In [63]:
model = Wo = Wo_inv = tokenizer = None
clear_mem()

In [64]:
{k: v.shape for k,v in ds_a2[0].items() if isinstance(v, torch.Tensor)}


{'acts-mlp.down_proj': torch.Size([1, 90, 2048]),
 'acts-self_attn': torch.Size([1, 90, 2048]),
 'acts-mlp.up_proj': torch.Size([1, 90, 6144]),
 'loss': torch.Size([]),
 'logits': torch.Size([151936]),
 'hidden_states': torch.Size([1, 90, 2048]),
 'attention_mask': torch.Size([90]),
 'label': torch.Size([]),
 'llm_ans': torch.Size([2]),
 'llm_log_prob_true': torch.Size([]),
 'supr_amounts': torch.Size([13, 1, 2048])}

In [65]:
ds2

Dataset({
    features: ['attention_mask', 'input_ids', 'label'],
    num_rows: 316
})

In [66]:
len(ds_a2), len(ds2)

(316, 316)

In [70]:
out_dir = Path('../data/activation_store')
name = acts_outfile.with_suffix("").relative_to('/tmp/activation_store')
acts_outfile2 = out_dir / name
acts_outfile2.parent.mkdir(parents=True, exist_ok=True)
acts_outfile2

PosixPath('../data/activation_store/ds_at-QwenQwen3-1.7B-truthfulQA-bool-train-316-90_v2')

In [67]:
from datasets import concatenate_datasets, load_dataset

ds_out = concatenate_datasets([ds_a2, ds2.select_columns('input_ids')], axis=1).with_format("torch")
ds_out.save_to_disk(acts_outfile2)
acts_outfile2

Saving the dataset (0/2 shards):   0%|          | 0/316 [00:00<?, ? examples/s]

PosixPath('/tmp/activation_store/ds_at-QwenQwen3-1.7B-truthfulQA-bool-train-316-90_v2.1')

In [68]:
f_config = acts_outfile2.with_suffix(".json")
json.dump({
    "model_name": model_name,
    "batch_size": batch_size,
    "max_length": max_length,
    'layer_groups': layer_groups,
    # 'model_config': model.config.to_dict(),
    "n_rows": len(ds_out),
}, open(f_config, "w"))
f_config

PosixPath('/tmp/activation_store/ds_at-QwenQwen3-1.7B-truthfulQA-bool-train-316-90_v2.json')

In [69]:
ds_out[-1]

{'acts-mlp.down_proj': tensor([[[ -0.5781,  -0.2930,  -0.3320,  ...,   0.6289,   1.4766,   0.3066],
          [ -0.5781,  -0.2930,  -0.3320,  ...,   0.6289,   1.4766,   0.3066],
          [ -0.5781,  -0.2930,  -0.3320,  ...,   0.6289,   1.4766,   0.3066],
          ...,
          [-14.2500,  -5.6562, -17.5000,  ...,  20.5000,   2.6250,   1.6484],
          [ -3.9375,  -0.9258,  -4.7188,  ...,   6.0312,   5.7188,   2.7656],
          [-14.0000,   2.2656,  -3.7188,  ...,   6.2812,  -3.4375,  15.4375]]]),
 'acts-self_attn': tensor([[[-0.4121,  5.3750,  2.3594,  ..., -2.8594, 12.2500, -8.0625],
          [-0.4121,  5.3750,  2.3594,  ..., -2.8594, 12.2500, -8.0625],
          [-0.4121,  5.3750,  2.3594,  ..., -2.8594, 12.2500, -8.0625],
          ...,
          [ 7.2500, -1.9219,  5.4062,  ...,  5.8750,  8.7500, -1.6875],
          [20.6250, -5.0000,  9.5625,  ...,  5.1875, 12.1875, -4.4062],
          [12.1250, -1.3359,  5.5000,  ..., -1.2891,  5.6562, -4.0938]]]),
 'acts-mlp.up_proj': ten